# CareerVista AI — Pakistan Career Guidance Prototype
## Google Colab notebook: explanations first, then code

**Goal:** build a learning prototype that compares resume text with student-friendly job-role descriptions, reports matched and missing skills, predicts broad categories, and visualizes the dataset.

**Data honesty:** the included 40-row CSV is curated prototype data, not verified live vacancies and not a representative sample of Pakistan's labor market. A relevant historical source is Open Data Pakistan's Pakistan's Job Market dataset (about 7,000 openings from Dec 2019–Mar 2021): https://www.opendata.com.pk/dataset/pakistan-s-job-market. Check the source license and date range before using it. Do not present old listings as current vacancies.

Run cells from top to bottom. This notebook demonstrates preprocessing, a baseline ML model, an MLP neural network, recommendation, evaluation, and export.

## Step 1 — Import libraries
We use pandas for data handling and scikit-learn for TF-IDF, classification, neural-network training, evaluation, and similarity.

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity

## Step 2 — Upload the dataset
Upload `careervista_pk_starter_dataset.csv` when the file picker appears.

In [ ]:
from google.colab import files
uploaded = files.upload()
DATA_FILE = "careervista_pk_starter_dataset.csv"
jobs = pd.read_csv(DATA_FILE)
print("Shape:", jobs.shape)
display(jobs.head())

## Step 3 — Inspect and clean data
Check missing values and duplicate job/category pairs before using data in a model.

In [ ]:
print(jobs.isna().sum())
jobs = jobs.drop_duplicates(subset=["job_title", "category"]).copy()
text_cols = ["job_title","category","experience_level","education",
             "required_skills","preferred_skills","job_description"]
for col in text_cols:
    jobs[col] = jobs[col].fillna("").astype(str).str.strip()
print("Clean shape:", jobs.shape)

## Step 4 — Prepare skills and text
Split skills only on semicolons, commas, or line breaks; do not split every slash because names such as CI/CD can contain slashes.

In [ ]:
def split_skills(value):
    if pd.isna(value):
        return []
    return sorted(set(
        re.sub(r"\\s+", " ", x.strip()).lower()
        for x in re.split(r"[;,|\\n]", str(value)) if x.strip()
    ))

jobs["required_skill_list"] = jobs["required_skills"].apply(split_skills)
jobs["model_text"] = (
    jobs["job_title"] + " " + jobs["category"] + " " +
    jobs["experience_level"] + " " + jobs["education"] + " " +
    jobs["required_skills"] + " " + jobs["preferred_skills"] + " " +
    jobs["job_description"]
).str.lower()
display(jobs[["job_title","category","required_skill_list"]].head())

## Step 5 — Train a baseline classifier
This predicts a broad job category from the job-role text. Because this starter set is small and several categories have few examples, its score is for demonstration only. It is not evidence of production accuracy.

In [ ]:
X, y = jobs["model_text"], jobs["category"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=1, stop_words="english")),
    ("model", LinearSVC())
])
baseline.fit(X_train, y_train)
pred = baseline.predict(X_test)
print("Baseline accuracy:", round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred, zero_division=0))

## Step 6 — Neural network model (MLP)
`MLPClassifier` is a feed-forward neural network. We compare it with the baseline. Small data may lead to unstable scores; report that limitation honestly.

In [ ]:
mlp = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=1,
                              stop_words="english", max_features=5000)),
    ("model", MLPClassifier(hidden_layer_sizes=(64,32), activation="relu",
                            max_iter=500, random_state=42))
])
mlp.fit(X_train, y_train)
mlp_pred = mlp.predict(X_test)
print("MLP accuracy:", round(accuracy_score(y_test, mlp_pred), 3))
print(classification_report(y_test, mlp_pred, zero_division=0))

## Step 7 — Resume-to-role recommendations
The score below is a heuristic, not a hiring probability. TF-IDF similarity ranks resume text against role text; skill coverage separately checks exact required-skill phrases. Exact matching can miss synonyms and context.

In [ ]:
def recommend_jobs(resume_text, jobs_df, top_n=5):
    resume_text = str(resume_text).lower()
    corpus = jobs_df["model_text"].fillna("").astype(str).tolist()
    vec = TfidfVectorizer(ngram_range=(1,2), stop_words="english")
    mat = vec.fit_transform(corpus + [resume_text])
    sims = cosine_similarity(mat[-1], mat[:-1]).ravel()
    out = jobs_df.copy()
    out["text_similarity"] = sims
    matched, missing, coverage = [], [], []
    for skills in out["required_skill_list"]:
        got = [s for s in skills if s in resume_text]
        gap = [s for s in skills if s not in resume_text]
        matched.append(got); missing.append(gap)
        coverage.append(len(got) / max(1, len(skills)))
    out["matched_skills"] = matched
    out["skills_to_develop"] = missing
    out["required_skill_coverage"] = coverage
    out["match_score"] = 0.7*out["text_similarity"] + 0.3*out["required_skill_coverage"]
    return out.sort_values("match_score", ascending=False).head(top_n)

sample_resume = "Computer science student. Python, SQL, Excel, pandas, data visualization, Git, basic statistics. University sales analysis project."
display(recommend_jobs(sample_resume, jobs)[[
    "job_title","category","match_score","matched_skills","skills_to_develop"
]])

## Step 8 — Try a resume (remove personal information first)
Paste sanitized resume text below. Do not paste phone number, home address, national ID, or other sensitive personal data into a shared notebook.

In [ ]:
resume_text = '''
Paste resume text here after removing personal details.
'''
display(recommend_jobs(resume_text, jobs, top_n=10)[[
    "job_title","category","experience_level","match_score",
    "matched_skills","skills_to_develop"
]])

## Step 9 — Visualize categories
This counts records in this CSV only. It is not a real-world demand chart.

In [ ]:
counts = jobs["category"].value_counts().sort_values()
plt.figure(figsize=(10,6))
counts.plot(kind="barh")
plt.title("Starter dataset: role records by category")
plt.xlabel("Number of records")
plt.tight_layout()
plt.show()

## Step 10 — Export files for Streamlit
Save the cleaned dataset and both trained models. Your Streamlit app can load these files, but the matching function must also be included in `app.py` or a Python module.

In [ ]:
clean = jobs.drop(columns=["required_skill_list","model_text"], errors="ignore")
clean.to_csv("careervista_jobs_clean.csv", index=False)
joblib.dump(baseline, "careervista_category_model.joblib")
joblib.dump(mlp, "careervista_mlp_model.joblib")
print("Exported cleaned CSV and two model files.")
files.download("careervista_jobs_clean.csv")
files.download("careervista_category_model.joblib")
files.download("careervista_mlp_model.joblib")

## Step 11 — Upgrade to research-quality Pakistan data
1. Download the historical Pakistan job-market dataset from Open Data Pakistan and read its license/attribution.
2. Inspect actual columns and missing values before mapping them.
3. Map available fields to title, description, city, experience and date; do not invent absent skills.
4. If extracting skills from descriptions, mark them as inferred and manually validate a sample.
5. Record the dataset's date range. Historical data are not current vacancies.
6. Evaluate on held-out data; if dates exist, consider a time-based split.
7. For a real student-facing service, add data provenance, privacy notice, feedback, and a clear note that recommendations are guidance, not hiring decisions.

## Submission checklist
- [ ] Colab notebook and output screenshots
- [ ] Cleaned CSV and source/license note
- [ ] Streamlit `app.py` and `requirements.txt`
- [ ] Evaluation results and limitations
- [ ] PDF/Word report with problem, method, results, screenshots, future work
- [ ] Public URL tested in an incognito/private browser